# Prozorro.Sale — 2nd place solution
*текст курсивом*
```
MyDrive/Prozzoro/
├── TRAIN_DATA/        train.csv, documents_manifest.csv, emb_train.npy
├── NEW_TEST/          test_phase2.csv, test_phase2_documents_manifest.csv,
│                      emb_test_phase2.npy, test_phase2_sample_submission.csv
├── RESULTS/           feature cache + per-model OOF/test predictions
└── final_submission/  submission_final.csv,submission_ensemble_equal.csv
```

**Competition:** predict the auction-participation class `target ∈ {0,1,2,3}` and maximise
**Quadratic Weighted Kappa (QWK)** on a *time-based* split.

**This is the consolidated, single-notebook version of the winning pipeline (2nd place, score 0.7118).**

It trains **everything from scratch** on `TRAIN_DATA/` and scores the **brand-new phase-2 test**
in `NEW_TEST/` (which ships its own fresh embeddings). Every expensive artifact is
cached to `RESULTS/` (features, per-model OOF/test predictions), so a second run is near-instant and never retrains from zero.

### The recipe that won
1. **Regress `log1p(n_bids)`** (a count) instead of classifying directly — then tune
   3 cut points that maximise QWK. Class 0 dominates (~66%), so this is more robust.
2. **Feature engineering**: base numerics + log transforms + booleans, rich **text**
   statistics, **time/seasonality**, **leakage-free** frequency / missing / ratio
   aggregates, **KOATUU** administrative hierarchy, **document-manifest** features, and
   **Qwen3-Embedding-4B** text vectors compressed with **PCA → 256 dims**.
3. **Three complementary learners**, each with out-of-fold (OOF) predictions on the
   same expanding **temporal folds** + a seed-averaged refit on all of train:
   - **LightGBM** — `regression_l2` on the log count (drops the two ultra-high-card IDs).
   - **XGBoost** — native `count:poisson` objective.
   - **CatBoost** — RMSE on the log count, *keeping* the high-card IDs (native cat support).
4. **Blend** the three with non-negative weights optimised on OOF + last-fold QWK, then
   pick the 3 thresholds on the most-recent fold (closest proxy to the future test).

### Inputs
- `TRAIN_DATA/`: `train.csv`, `documents_manifest.csv`, `emb_train.npy`
- `NEW_TEST/`: `test_phase2.csv`, `test_phase2_documents_manifest.csv`, `emb_test_phase2.npy`

### Outputs (written to `RESULTS/` and `final_submission/`)
- `features_train.parquet`, `features_test.parquet`, `train_meta.parquet`, `test_meta.parquet`, `meta.json`
- `lgb_*`, `xgb_*`, `cb_*` `_oof/_test.parquet` prediction caches
- `submission_final.csv` (optimised blend) and `submission_ensemble_equal.csv`

## 0. Setup — imports, paths, configuration

All knobs live here. `REUSE_SAVED=True` makes the notebook reuse any cached artifact in `RESULTS/` instead of recomputing it.

In [1]:
import os, re, gc, json, time, sys, subprocess

# --- mount Google Drive ------------------------------------------------------
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Root of the project inside your Drive. Change "Prozzoro" if your folder differs.
# (The original Colab notebooks used "Kaggle_Prozzoro" — set whichever you actually have.)
DRIVE_BASE = "/content/drive/MyDrive/Prozzoro"

# Install any missing model libraries (Colab usually ships lightgbm/xgboost; catboost may not).
for pkg in ("lightgbm", "xgboost", "catboost", "pyarrow"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor, Pool
from scipy.optimize import minimize
from collections import Counter
from itertools import combinations

# --- identical folder layout to the local project ---------------------------
ROOT      = DRIVE_BASE
TRAIN_DIR = os.path.join(ROOT, "TRAIN_DATA")
TEST_DIR  = os.path.join(ROOT, "NEW_TEST")
RESULTS   = os.path.join(ROOT, "RESULTS")
SUB_DIR   = os.path.join(ROOT, "final_submission")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(SUB_DIR, exist_ok=True)
assert os.path.isdir(ROOT), f"Drive folder not found: {ROOT}  (check DRIVE_BASE / your folder name)"
assert os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR), (
    f"Upload TRAIN_DATA/ and NEW_TEST/ into {ROOT}")

# --- GPU autodetection (Colab T4 -> big speedup for XGBoost/CatBoost) --------
_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
HAS_GPU       = bool(_gpu)
XGB_DEVICE    = "cuda" if HAS_GPU else "cpu"
CAT_TASK_TYPE = "GPU" if HAS_GPU else "CPU"
print("GPU:", _gpu or "NONE (CPU) -> set Runtime > Change runtime type > GPU (T4) for speed")

# --- configuration -----------------------------------------------------------
REUSE_SAVED   = True
PCA_DIM       = 256
N_FOLDS       = 3
HOLDOUT_FRAC  = 0.08
SEEDS         = [42, 7, 123]
HIGH_CARD_COLS = ("organizer_id", "koatuu")
NUM_CLASS     = 4

# Train only or Train + Test PCA fit
PCA_FREQ_TRAIN_ONLY = True

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__)
print("ROOT:", ROOT, "| XGB_DEVICE:", XGB_DEVICE, "| CAT_TASK_TYPE:", CAT_TASK_TYPE)
print("PCA/freq fit on:", "TRAIN ONLY" if PCA_FREQ_TRAIN_ONLY else "TRAIN+TEST")
_t0 = time.time()
def log(*a): print(f"[{time.time()-_t0:6.1f}s]", *a, flush=True)

Mounted at /content/drive
GPU: NVIDIA L4
lightgbm 4.6.0 | xgboost 3.3.0
ROOT: /content/drive/MyDrive/Prozzoro | XGB_DEVICE: cuda | CAT_TASK_TYPE: GPU
PCA/freq fit on: TRAIN ONLY


## 1. Validation harness — QWK, threshold tuning, temporal folds

The competition target buckets the raw participant count `n_bids` into 4 classes:
`0→0 bids`, `1→1 bid`, `2→2-4 bids`, `3→5+ bids`. We regress the log count, then search
3 cut points that maximise QWK (Nelder-Mead). Validation uses **expanding temporal
folds**: each fold trains on strictly-earlier months and validates on the next window,
mirroring the future test. The **last fold == the teacher's single 8% holdout**, so it
is our closest proxy to leaderboard behaviour.

In [ ]:
def to_class(n):
    n = np.asarray(n)
    return np.where(n <= 0, 0, np.where(n == 1, 1, np.where(n <= 4, 2, 3))).astype(int)

def qwk(a, b, k=NUM_CLASS):
    a = np.asarray(a, int); b = np.asarray(b, int)
    O = np.zeros((k, k), float)
    for x, y in zip(a, b): O[x, y] += 1
    w = np.array([[(i - j) ** 2 / (k - 1) ** 2 for j in range(k)] for i in range(k)], float)
    E = np.outer(O.sum(1), O.sum(0)) / max(O.sum(), 1.0)
    return 1.0 - (w * O).sum() / max((w * E).sum(), 1e-9)

def round_with(thr, p):
    thr = np.sort(np.asarray(thr, float)); p = np.asarray(p, float)
    return np.clip(sum((p > t).astype(int) for t in thr), 0, NUM_CLASS - 1)

def tune_thresholds(yc, p, init=(0.5, 1.5, 4.5), maxiter=2000):
    # find 3 sorted cut points maximising QWK of round_with(thr, p)
    yc = np.asarray(yc, int); p = np.asarray(p, float)
    r = minimize(lambda t: -qwk(yc, round_with(t, p)), np.asarray(init, float),
                 method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-4, "maxiter": maxiter})
    thr = np.sort(r.x)
    return thr, qwk(yc, round_with(thr, p))

print("validation harness ready (to_class / qwk / round_with / tune_thresholds)")

validation harness ready (to_class / qwk / round_with / tune_thresholds)


## 2. Feature engineering

If a feature cache already exists in `RESULTS/`, we load it and **skip all of section 2**.
Otherwise we build the full matrix from the raw files (train + the brand-new phase-2 test)
and cache it. The same transforms are applied to both sides so the schemas line up exactly.

In [ ]:
FEATURE_CACHE = all(os.path.exists(os.path.join(RESULTS, f)) for f in
                    ["features_train.parquet", "features_test.parquet",
                     "train_meta.parquet", "test_meta.parquet", "meta.json"])
BUILD_FEATURES = not (REUSE_SAVED and FEATURE_CACHE)
print("Feature cache present:", FEATURE_CACHE, "| will (re)build features:", BUILD_FEATURES)

Feature cache present: True | will (re)build features: False


### 2.0 Load raw train / test

In [ ]:
if BUILD_FEATURES:
    log("loading raw csv ...")
    train = pd.read_csv(os.path.join(TRAIN_DIR, "train.csv"), low_memory=False).reset_index(drop=True)
    test  = pd.read_csv(os.path.join(TEST_DIR, "test_phase2.csv"), low_memory=False).reset_index(drop=True)
    log("train", train.shape, "| test", test.shape)
    print("target distribution:", train["target"].value_counts(normalize=True).sort_index().round(3).to_dict())

### 2.1 KOATUU administrative hierarchy

`koatuu` is Ukraine's 10-digit territorial code; its prefixes encode oblast → raion →
council → settlement. Decoding gives coarser geographic groupings that generalise far
better than the raw, very-high-cardinality code. (No target used → safe on train+test.)

In [ ]:
def add_koatuu_features(train_df, test_df):
    full = pd.concat([train_df["koatuu"], test_df["koatuu"]], ignore_index=True)
    code_ = pd.to_numeric(full, errors="coerce").astype("Int64").astype("string").str.zfill(10).fillna("")
    def level_of(c):
        if c == "":                return "missing"
        if c.endswith("00000000"): return "oblast"
        if c.endswith("00000"):    return "raion"
        if c.endswith("00"):       return "council"
        return "settlement"
    feats = pd.DataFrame(index=full.index)
    feats["koatuu_raion"]        = code_.str[:5].where(code_ != "", "NA")
    feats["koatuu_council"]      = code_.str[:8].where(code_ != "", "NA")
    feats["koatuu_level"]        = code_.map(level_of)
    feats["is_settlement_level"] = ((code_ != "") & (~code_.str.endswith("00"))).astype("int8")
    cat_cols, num_cols = ["koatuu_raion", "koatuu_council", "koatuu_level"], ["is_settlement_level"]
    tr = feats.iloc[:len(train_df)].copy(); tr.index = train_df.index
    te = feats.iloc[len(train_df):].copy(); te.index = test_df.index
    return pd.concat([train_df, tr], axis=1), pd.concat([test_df, te], axis=1), cat_cols, num_cols

if BUILD_FEATURES:
    train, test, KOATUU_CAT, KOATUU_NUM = add_koatuu_features(train, test)
    log("KOATUU features:", KOATUU_CAT + KOATUU_NUM)

### 2.2 Document-manifest features

`documents_manifest.csv` has one row per uploaded document. We aggregate it to one row
per auction (key: `manifest.procedure_id` ↔ `train/test.id`): document counts, distinct
types/formats, **photo** statistics (**our hypothesis: people like photos and may bid more on well-illustrated lots**), per-type
counts/presence flags, and pairwise presence combos for the top-8 document types.

> Correctness detail: the top-8 types for the combos are derived from **train** and the
> *same* set is reused for test, so the combo columns mean the same thing on both sides.

In [ ]:
MANIFEST_KEY, JOIN_KEY = "procedure_id", "id"
TOP_K_DOCTYPES_FOR_PAIRS = 8
IMAGE_EXT = {"jpg", "jpeg", "png", "gif", "bmp", "webp", "tif", "tiff"}

def build_manifest_features(manifest_path, top_pres=None):
    man = pd.read_csv(manifest_path, usecols=["procedure_id", "documentType", "ext"], low_memory=False)
    man["is_image"] = man["ext"].fillna("").str.lower().isin(IMAGE_EXT)
    feats = man.groupby(MANIFEST_KEY).agg(
        m_documents_total=("documentType", "count"),
        m_unique_doctypes=("documentType", "nunique"),
        m_unique_formats=("ext", "nunique"),
        m_images_total=("is_image", "sum"),
    )
    feats["m_image_ratio"]    = feats["m_images_total"] / feats["m_documents_total"].clip(lower=1)
    feats["m_log_images"]     = np.log1p(feats["m_images_total"])
    feats["m_non_image_docs"] = feats["m_documents_total"] - feats["m_images_total"]
    for k in (1, 3, 5, 10, 20):
        feats[f"m_photos_ge_{k}"] = (feats["m_images_total"] >= k).astype("int8")
    type_counts = man.pivot_table(index=MANIFEST_KEY, columns="documentType",
                                  values="ext", aggfunc="count", fill_value=0)
    type_counts.columns = [f"mdoc_{c}" for c in type_counts.columns]
    feats = feats.join(type_counts)
    pres = (type_counts > 0).astype("int8")
    pres.columns = [c.replace("mdoc_", "mpres_") for c in type_counts.columns]
    feats = feats.join(pres)
    if top_pres is None:
        top_types = type_counts.sum().sort_values(ascending=False).head(TOP_K_DOCTYPES_FOR_PAIRS)
        top_pres = [c.replace("mdoc_", "mpres_") for c in top_types.index]
    for c in top_pres:
        if c not in pres.columns:
            pres[c] = np.int8(0)
    for a, b in combinations(top_pres, 2):
        feats[f"{a}_AND_{b}"] = (pres[a].values & pres[b].values).astype("int8")
    feats.index.name = MANIFEST_KEY
    return feats.reset_index(), top_pres

def merge_manifest(df, mf, cols):
    df = df.merge(mf, left_on=JOIN_KEY, right_on=MANIFEST_KEY, how="left")
    df = df.drop(columns=[MANIFEST_KEY])
    df[cols] = df[cols].fillna(0)
    imgs = pd.to_numeric(df["m_images_total"], errors="coerce").fillna(0)
    df = pd.concat([df, pd.DataFrame({
        "m_photos_per_item": (imgs / (pd.to_numeric(df["n_items"], errors="coerce") + 1)).astype("float32"),
        "m_photos_per_doc":  (imgs / (pd.to_numeric(df["n_documents"], errors="coerce") + 1)).astype("float32"),
    }, index=df.index)], axis=1)
    return df

if BUILD_FEATURES:
    log("building manifest features (train) ...")
    mf_train, TOP_PRES = build_manifest_features(os.path.join(TRAIN_DIR, "documents_manifest.csv"))
    log("building manifest features (test) ...")
    mf_test, _ = build_manifest_features(os.path.join(TEST_DIR, "test_phase2_documents_manifest.csv"), top_pres=TOP_PRES)
    MANIFEST_COLS = [c for c in sorted(set(mf_train.columns) | set(mf_test.columns)) if c != MANIFEST_KEY]
    for mf in (mf_train, mf_test):
        for c in MANIFEST_COLS:
            if c not in mf.columns: mf[c] = 0
    train = merge_manifest(train, mf_train, MANIFEST_COLS)
    test  = merge_manifest(test,  mf_test,  MANIFEST_COLS)
    MANIFEST_COLS = MANIFEST_COLS + ["m_photos_per_item", "m_photos_per_doc"]
    log("manifest features:", len(MANIFEST_COLS))
    del mf_train, mf_test; gc.collect()

### 2.3 Text statistics

For each of `title`, `description`, `items_text` we compute cheap surface statistics:
length, character-class ratios, sentence/word counts, lexical diversity, hapax ratio and
Shannon entropy.

**Our hypothesis: richer, more structured listings tend to attract more bidders, and these are a strong proxy before the heavier semantic embeddings come in.**

In [ ]:
TEXT_COLS = ["title", "description", "items_text"]
_WORD_RE  = re.compile(r"\w+", re.UNICODE)
_URL_RE   = re.compile(r"https?://")
_EMAIL_RE = re.compile(r"\S+@\S+")
_PCT_RE   = re.compile(r"\d+\s*%")
_MONEY_RE = re.compile(r"\d[\d\s.,]*\s*(?:грн|uah|usd|eur|₴|\$|€)", re.IGNORECASE)

def compute_text_features(series, prefix):
    s = series.fillna("").astype(str)
    chars = s.str.len()
    out = pd.DataFrame(index=s.index)
    out["chars"]           = chars
    out["digit_ratio"]     = s.str.count(r"\d") / chars.clip(lower=1)
    out["uppercase_ratio"] = s.str.count(r"[A-ZА-ЯЀ-ЯЇІЄҐ]") / s.str.count(r"[^\W\d_]").clip(lower=1)
    out["space_ratio"]     = s.str.count(r"\s") / chars.clip(lower=1)
    out["punct_ratio"]     = s.str.count(r"[^\w\s]") / chars.clip(lower=1)
    out["sentence_count"]  = s.str.count(r"[.!?]").clip(lower=1)
    out["url_count"]       = s.str.count(_URL_RE)
    out["email_count"]     = s.str.count(_EMAIL_RE)
    out["percent_count"]   = s.str.count(_PCT_RE)
    out["money_count"]     = s.str.count(_MONEY_RE)
    n = len(s)
    words = np.zeros(n, np.int32); avg_wl = np.zeros(n); med_wl = np.zeros(n); max_wl = np.zeros(n)
    lexdiv = np.zeros(n); reps = np.zeros(n); hapax = np.zeros(n); entropy = np.zeros(n)
    for i, text in enumerate(s.values):
        toks = _WORD_RE.findall(text); w = len(toks); words[i] = w
        if w == 0: continue
        lens = [len(t) for t in toks]
        avg_wl[i] = sum(lens) / w; med_wl[i] = float(np.median(lens)); max_wl[i] = max(lens)
        freq = Counter(t.lower() for t in toks); u = len(freq)
        lexdiv[i] = u / w; reps[i] = 1.0 - u / w
        hapax[i] = sum(1 for v in freq.values() if v == 1) / u
        cnt = np.fromiter(freq.values(), float, u); p = cnt / cnt.sum()
        entropy[i] = float(-(p * np.log2(p)).sum())
    out["words"] = words; out["avg_word_len"] = avg_wl
    out["median_word_len"] = med_wl; out["max_word_len"] = max_wl
    out["lexical_diversity"] = lexdiv; out["repetition_ratio"] = reps
    out["hapax_ratio"] = hapax; out["shannon_entropy"] = entropy
    out["avg_sentence_len"] = out["words"] / out["sentence_count"].clip(lower=1)
    return out.add_prefix(f"{prefix}_").astype("float32")

def add_text_features(df):
    blocks = [compute_text_features(df[c], c) for c in TEXT_COLS]
    cols = [c for b in blocks for c in b.columns]
    return pd.concat([df] + blocks, axis=1), cols

if BUILD_FEATURES:
    log("text features (train) ...")
    train, TEXT_FEAT_COLS = add_text_features(train)
    log("text features (test) ...")
    test, _ = add_text_features(test)
    log("text features:", len(TEXT_FEAT_COLS))

### 2.4 Time / seasonality

There is no full calendar date — only `auction_year`, `auction_month` and four duration
columns. We build cyclical month encodings, quarter/season, a monotonic `ym_index` drift
signal, and schedule interactions between the duration columns.

In [ ]:
def add_time_features(df):
    month = df["auction_month"].astype(float); year = df["auction_year"].astype(float)
    pub  = pd.to_numeric(df.get("publish_to_auction_days"), errors="coerce")
    tend = pd.to_numeric(df.get("tender_days"), errors="coerce")
    enq  = pd.to_numeric(df.get("enquiry_days"), errors="coerce")
    feats = pd.DataFrame(index=df.index)
    feats["month_sin"] = np.sin(2 * np.pi * month / 12.0)
    feats["month_cos"] = np.cos(2 * np.pi * month / 12.0)
    feats["quarter"]   = (month - 1) // 3 + 1
    feats["season"]    = month % 12 // 3
    feats["ym_index"]  = year * 12 + month
    feats["tender_share_of_window"]  = (tend / pub.replace(0, np.nan)).clip(0, 1)
    feats["enquiry_share_of_window"] = (enq / pub.replace(0, np.nan)).clip(0, 1)
    feats["lead_time_total"] = pub.fillna(0) + tend.fillna(0)
    feats = feats.astype("float32")
    return pd.concat([df, feats], axis=1), feats.columns.tolist()

if BUILD_FEATURES:
    train, TIME_FEAT_COLS = add_time_features(train)
    test, _ = add_time_features(test)
    log("time features:", TIME_FEAT_COLS)

### 2.5 Leakage-free frequency / geo aggregates, missing flags, ratios

Frequency counts for high-volume keys, missing-value indicators for columns the EDA
flagged as missing-not-at-random, and a few simple ratios. These use **no target**.

The frequency maps follow the `PCA_FREQ_TRAIN_ONLY` flag set in the setup cell. When
**train-only**, test rows are *mapped* through the train maps (unseen keys → NaN, which the
trees handle), so the cross-validation is invariant to the test set; when **train+test**,
it reproduces the original winning fit.

In [ ]:
FREQ_COLS    = ["organizer_id", "region", "koatuu", "owner"]
MISSING_COLS = ["land_area", "dutchStep_amount", "dutchStep_pct",
                "registrationFee_amount", "koatuu", "organizer_id"]

def _build_leakfree(df, freq_maps):
    feats = pd.DataFrame(index=df.index)
    for col in FREQ_COLS:
        feats[f"freq_{col}"] = df[col].astype("string").fillna("NA").map(freq_maps[col]).astype("float32")
    for col in MISSING_COLS:
        feats[f"{col}_is_missing"] = df[col].isna().astype("int8")
    n_items = pd.to_numeric(df["n_items"], errors="coerce")
    n_docs  = pd.to_numeric(df["n_documents"], errors="coerce")
    tdays   = pd.to_numeric(df["tender_days"], errors="coerce")
    val     = pd.to_numeric(df["value_amount"], errors="coerce")
    feats["docs_per_item"]  = (n_docs / (n_items + 1)).astype("float32")
    feats["docs_per_day"]   = (n_docs / (tdays + 1)).astype("float32")
    feats["value_per_item"] = (val / (n_items + 1)).astype("float32")
    return feats

def add_leakfree_aggregates(train_df, test_df):
    # Frequency maps: train-only (option A) or train+test (option B), per FIT_SCOPE.
    if PCA_FREQ_TRAIN_ONLY:
        src = train_df[FREQ_COLS]
    else:
        src = pd.concat([train_df[FREQ_COLS], test_df[FREQ_COLS]], ignore_index=True)
    freq_maps = {c: src[c].astype("string").fillna("NA").value_counts() for c in FREQ_COLS}
    tr, te = _build_leakfree(train_df, freq_maps), _build_leakfree(test_df, freq_maps)
    return pd.concat([train_df, tr], axis=1), pd.concat([test_df, te], axis=1), tr.columns.tolist()

if BUILD_FEATURES:
    train, test, AGG_FEAT_COLS = add_leakfree_aggregates(train, test)
    log("leak-free aggregates:", len(AGG_FEAT_COLS))

### 2.6 Log transforms, boolean casts, and the final feature lists

We log1p the heavy-tailed money/area columns and cast booleans to float. Then we define
the numeric and categorical feature lists. `organizer_id` and `koatuu` are kept in the
matrix (CatBoost uses them natively); LightGBM/XGBoost drop them later because their
high cardinality invites memorisation.

In [ ]:
BASE_NUM = ["value_amount", "minimalStep_amount", "registrationFee_amount", "guarantee_amount",
            "dutchStep_amount", "dutchStep_pct", "tenderAttempts", "n_items", "n_documents",
            "enquiry_days", "tender_days", "publish_to_auction_days", "tenderend_to_auction_days",
            "land_area", "minstep_ratio", "regfee_ratio", "guarantee_ratio",
            "doc_notice", "doc_technicalSpecifications", "doc_illustration", "doc_contractProforma",
            "doc_evaluationCriteria", "doc_clarifications", "doc_digitalSignature", "doc_x_itemPlan",
            "doc_other", "auction_year", "auction_month"]
BASE_BOOL = ["vat_included", "has_address", "has_encumbrances", "has_joint_ownership", "has_utilities"]
BASE_CAT  = ["sellingMethod", "family", "format", "region", "koatuu", "itemPropsType",
             "ownershipType", "cav_div", "currency", "owner", "organizer_id", "organizer_region"]
LOGV     = ["value_amount", "minimalStep_amount", "registrationFee_amount", "guarantee_amount", "land_area"]
LOG_COLS = [f"log_{c}" for c in LOGV]

def _add_logs(df):
    block = pd.DataFrame(
        {f"log_{c}": np.log1p(pd.to_numeric(df[c], errors="coerce").clip(lower=0)).astype("float32") for c in LOGV},
        index=df.index)
    return pd.concat([df, block], axis=1)

if BUILD_FEATURES:
    train = _add_logs(train); test = _add_logs(test)
    for df in (train, test):
        for c in BASE_BOOL:
            df[c] = df[c].astype("float32")
    SIMPLE_NUM = (BASE_NUM + LOG_COLS + BASE_BOOL + TEXT_FEAT_COLS + TIME_FEAT_COLS
                  + AGG_FEAT_COLS + KOATUU_NUM)
    NUM_FINAL  = SIMPLE_NUM + MANIFEST_COLS
    ALL_CAT    = BASE_CAT + KOATUU_CAT          # incl. organizer_id, koatuu
    log("numeric features:", len(NUM_FINAL), "| categorical:", len(ALL_CAT))

### 2.7 Embeddings → PCA-256

The offline Qwen3-Embedding-4B vectors (2560-d, float16) carry the semantic signal of the listing text. Gradient-boosted trees gain nothing from 2560 redundant axes, so we compress to 256 principal components — essentially all the signal, ~10× faster, slightly denoised.
PCA is unsupervised and follows the same `PCA_FREQ_TRAIN_ONLY` flag.


**Surprisingly, larger number of components (384, for example) did not improve the solution.**

PCA has to be fit on some set of rows, then applied to transform every row. The question is which rows you fit it on.

We submiited 2 options:

**Train-only PCA (submission_final_1, the 2nd-place file)**

pca.fit(train_embeddings) → learn axes from the training rows only.
pca.transform(train) and pca.transform(test) → apply those fixed axes to both.
The test set never influences the components. Same idea applied to the frequency encodings (counts computed on train only).

**Train+test PCA (submission_final_2, no place solution)**

pca.fit(concat(train_embeddings, test_embeddings)) → learn axes from train and test rows together. Then transform both.

In [ ]:
if BUILD_FEATURES:
    from sklearn.decomposition import PCA
    log("loading embeddings ...")
    emb_tr = np.load(os.path.join(TRAIN_DIR, "emb_train.npy")).astype("float32")
    emb_te = np.load(os.path.join(TEST_DIR, "emb_test_phase2.npy")).astype("float32")
    assert emb_tr.shape[0] == len(train) and emb_te.shape[0] == len(test), "embedding row mismatch"
    n_comp = min(PCA_DIM, emb_tr.shape[1])
    pca = PCA(n_components=n_comp, svd_solver="randomized", random_state=42)

    #flag
    if PCA_FREQ_TRAIN_ONLY:
        log(f"PCA fit on TRAIN {emb_tr.shape[0]} x {emb_tr.shape[1]} -> {n_comp} (test projected) ...")
        pca.fit(emb_tr)                      # train-only basis -> CV invariant to test
    else:
        log(f"PCA fit on TRAIN+TEST {emb_tr.shape[0]+emb_te.shape[0]} x {emb_tr.shape[1]} -> {n_comp} ...")
        pca.fit(np.vstack([emb_tr, emb_te]))
    EMB_TRAIN = pca.transform(emb_tr).astype("float32")
    EMB_TEST  = pca.transform(emb_te).astype("float32")
    log(f"PCA explained variance = {pca.explained_variance_ratio_.sum():.3f}")
    del emb_tr, emb_te; gc.collect()

### 2.8 Assemble the unified feature matrix and cache it

We concatenate numeric + categorical (as strings) + the 256 PCA dims into one frame for
train and the new test, assert identical columns, and write everything to `RESULTS/`.
After this cell runs once, every later run loads the cache and skips section 2 entirely.

In [ ]:
if BUILD_FEATURES:
    def assemble(df, emb):
        num = df[[c for c in NUM_FINAL if c in df.columns]].apply(pd.to_numeric, errors="coerce").astype("float32")
        cat = df[ALL_CAT].astype("string").fillna("NA").astype(str)
        emb_df = pd.DataFrame(np.asarray(emb, "float32"),
                              columns=[f"emb_{i}" for i in range(emb.shape[1])], index=df.index)
        return pd.concat([num, cat, emb_df], axis=1)

    Xtr = assemble(train, EMB_TRAIN)
    Xte = assemble(test, EMB_TEST)
    assert list(Xtr.columns) == list(Xte.columns), "train/test column mismatch"
    ym_arr = (train["auction_year"].astype(int) * 12 + train["auction_month"].astype(int)).to_numpy()
    counts = train["n_bids"].to_numpy()

    Xtr.to_parquet(os.path.join(RESULTS, "features_train.parquet"))
    Xte.to_parquet(os.path.join(RESULTS, "features_test.parquet"))
    pd.DataFrame({"n_bids": counts, "y_log": np.log1p(counts), "ym": ym_arr}).to_parquet(
        os.path.join(RESULTS, "train_meta.parquet"))
    pd.DataFrame({"id": test["id"].to_numpy()}).to_parquet(os.path.join(RESULTS, "test_meta.parquet"))
    json.dump({"cat_features": ALL_CAT, "n_folds": N_FOLDS, "emb_dim": int(EMB_TRAIN.shape[1]),
               "n_features": int(Xtr.shape[1]), "high_card_cols": list(HIGH_CARD_COLS)},
              open(os.path.join(RESULTS, "meta.json"), "w"), indent=2)
    log("cached features ->", RESULTS, "| Xtr", Xtr.shape, "| Xte", Xte.shape)
    del Xtr, Xte, train, test; gc.collect()

## 3. Load the cached feature matrix

The single source of truth for the modeling stage. Loaded from `RESULTS/` whether it was just built or cached from a previous run.

In [ ]:
meta = json.load(open(os.path.join(RESULTS, "meta.json")))
CAT_ALL = list(meta["cat_features"])
N_FOLDS = int(meta["n_folds"])

Xtr_raw  = pd.read_parquet(os.path.join(RESULTS, "features_train.parquet"))
Xte_raw  = pd.read_parquet(os.path.join(RESULTS, "features_test.parquet"))
tmeta    = pd.read_parquet(os.path.join(RESULTS, "train_meta.parquet"))
test_meta = pd.read_parquet(os.path.join(RESULTS, "test_meta.parquet"))

counts_all = tmeta["n_bids"].to_numpy().astype(float)
y_log_all  = np.log1p(counts_all)
ym         = tmeta["ym"].to_numpy()
test_ids   = test_meta["id"].to_numpy()
log("features", Xtr_raw.shape, "| test", Xte_raw.shape, "| cat_features", len(CAT_ALL))
print("target class distribution:",
      pd.Series(to_class(counts_all)).value_counts(normalize=True).sort_index().round(4).to_dict())

[  16.9s] features (273804, 486) | test (2100, 486) | cat_features 15
target class distribution: {0: 0.6627, 1: 0.1395, 2: 0.1615, 3: 0.0363}


### 3.1 Expanding temporal folds (shared by all models)

Fold boundaries come from month-quantiles of the train set; the last fold is the most-recent 8% window. The exact same masks are used for LightGBM, XGBoost and CatBoost so their OOF predictions are perfectly aligned for blending.

In [ ]:
qs   = sorted(1.0 - HOLDOUT_FRAC * k for k in range(N_FOLDS + 1))
cuts = [np.quantile(ym, q) for q in qs]
def fold_masks(k):
    lo, hi = cuts[k], cuts[k + 1]
    fm = ym < lo
    vm = (ym >= lo) if k == N_FOLDS - 1 else ((ym >= lo) & (ym < hi))
    return fm, vm
N_LAST = int(fold_masks(N_FOLDS - 1)[1].sum())
print("fold val sizes:", [int(fold_masks(k)[1].sum()) for k in range(N_FOLDS)], "| last-fold:", N_LAST)

fold val sizes: [17480, 20625, 31618] | last-fold: 31618


### 3.2 Per-model feature matrices

- **LGBM / XGBoost**: drop the two ultra-high-card IDs; cast the rest of the categoricals
  to a shared `category` dtype built on the train+test union (so codes match at inference).
- **CatBoost**: keep every categorical (including the IDs) as strings — its ordered
  target statistics handle high cardinality safely.

In [ ]:
def make_cat_dtype(a, b):
    vals = sorted(set(a.astype(str).fillna("NA")) | set(b.astype(str).fillna("NA")))
    return pd.CategoricalDtype(vals)

def build_lgb_xgb():
    Xtr, Xte = Xtr_raw.copy(), Xte_raw.copy()
    drop = [c for c in HIGH_CARD_COLS if c in Xtr.columns]
    Xtr = Xtr.drop(columns=drop); Xte = Xte.drop(columns=drop)
    cats = [c for c in CAT_ALL if c in Xtr.columns]
    for c in cats:
        dt = make_cat_dtype(Xtr[c], Xte[c])
        Xtr[c] = Xtr[c].astype(str).fillna("NA").astype(dt)
        Xte[c] = Xte[c].astype(str).fillna("NA").astype(dt)
    return Xtr, Xte, cats

def build_cb():
    Xtr, Xte = Xtr_raw.copy(), Xte_raw.copy()
    cats = [c for c in CAT_ALL if c in Xtr.columns]
    for c in cats:
        Xtr[c] = Xtr[c].astype(str).fillna("NA")
        Xte[c] = Xte[c].astype(str).fillna("NA")
    return Xtr, Xte, cats

Xtr_lx, Xte_lx, CATS_LX = build_lgb_xgb()
Xtr_cb, Xte_cb, CATS_CB = build_cb()
print("LGB/XGB", Xtr_lx.shape, "cats", len(CATS_LX), "| CatBoost", Xtr_cb.shape, "cats", len(CATS_CB))

LGB/XGB (273804, 484) cats 13 | CatBoost (273804, 486) cats 15


## 4. Train the three models (with caching)

Each model produces:
- **OOF predictions** over the temporal folds (used to fairly score and to fit blend weights/thresholds),
- a **seed-averaged refit** on all of train for the test predictions.

Predictions are cached as `RESULTS/{tag}_oof.parquet` and `RESULTS/{tag}_test.parquet`
(`lgb`, `xgb`, `cb`). If a cache exists and `REUSE_SAVED=True`, that model is skipped.

In [ ]:
def _paths(tag): return (os.path.join(RESULTS, f"{tag}_oof.parquet"),
                         os.path.join(RESULTS, f"{tag}_test.parquet"))
def has_saved(tag):
    op, tp = _paths(tag); return os.path.exists(op) and os.path.exists(tp)
def load_saved(tag):
    op, tp = _paths(tag); o = pd.read_parquet(op); t = pd.read_parquet(tp)
    col = "test" if "test" in t.columns else t.columns[0]
    return {"tag": tag, "oof_true": o["oof_true"].to_numpy(),
            "oof_pred": o["oof_pred"].to_numpy(), "test": t[col].to_numpy()}
def save_preds(tag, oof_true, oof_pred, test_pred):
    op, tp = _paths(tag)
    pd.DataFrame({"oof_true": oof_true, "oof_pred": oof_pred}).to_parquet(op)
    pd.DataFrame({"test": test_pred}).to_parquet(tp)
    log(f"saved {tag}_oof.parquet / {tag}_test.parquet")
def get_model(tag, trainer):
    if REUSE_SAVED and has_saved(tag):
        log(f"reusing cached predictions: {tag}"); return load_saved(tag)
    return trainer()

### 4.1 Model 1 — LightGBM (`regression_l2` on log count)

In [ ]:
LGB_PARAMS = dict(objective="regression_l2", n_estimators=4000, learning_rate=0.03,
                  num_leaves=127, min_child_samples=50, subsample=1.0, subsample_freq=0,
                  colsample_bytree=0.6, reg_lambda=20.0, reg_alpha=0.0, n_jobs=-1, verbosity=-1)

def train_lgb():
    oof_true, oof_pred, iters = [], [], []
    for k in range(N_FOLDS):
        fm, vm = fold_masks(k)
        m = lgb.LGBMRegressor(random_state=42, **LGB_PARAMS)
        m.fit(Xtr_lx.loc[fm], y_log_all[fm], categorical_feature=CATS_LX,
              eval_set=[(Xtr_lx.loc[vm], y_log_all[vm])], eval_metric="l2",
              callbacks=[lgb.early_stopping(80), lgb.log_evaluation(0)])
        bi = int(m.best_iteration_ or LGB_PARAMS["n_estimators"]); iters.append(bi)
        pc = np.clip(np.expm1(m.predict(Xtr_lx.loc[vm], num_iteration=bi)), 0, None)
        oof_true.append(counts_all[vm]); oof_pred.append(pc)
        _, q = tune_thresholds(to_class(counts_all[vm]), pc)
        log(f"[LGBM] fold {k+1}/{N_FOLDS} best_iter={bi} QWK={q:.4f}")
    oof_true, oof_pred = np.concatenate(oof_true), np.concatenate(oof_pred)
    n_best = int(np.median(iters) * 1.10); log("[LGBM] refit n_estimators", n_best)
    tp = np.zeros(len(Xte_lx))
    for sd in SEEDS:
        p = dict(LGB_PARAMS); p["n_estimators"] = n_best
        m = lgb.LGBMRegressor(random_state=sd, **p)
        m.fit(Xtr_lx, y_log_all, categorical_feature=CATS_LX)
        tp += np.expm1(m.predict(Xte_lx))
    save_preds("lgb", oof_true, oof_pred, np.clip(tp / len(SEEDS), 0, None))
    return load_saved("lgb")

best_lgb = get_model("lgb", train_lgb)

[  18.8s] reusing cached predictions: lgb


### 4.2 Model 2 — XGBoost (`count:poisson`)

In [ ]:
XGB_PARAMS = dict(objective="count:poisson", n_estimators=4000, learning_rate=0.03,
                  max_depth=8, subsample=0.8, colsample_bytree=0.5, reg_lambda=40.0,
                  reg_alpha=5.0, min_child_weight=5, tree_method="hist", device=XGB_DEVICE,
                  enable_categorical=True, eval_metric="poisson-nloglik", n_jobs=-1)

def train_xgb():
    oof_true, oof_pred, iters = [], [], []
    for k in range(N_FOLDS):
        fm, vm = fold_masks(k)
        m = xgb.XGBRegressor(random_state=42, early_stopping_rounds=80, **XGB_PARAMS)
        m.fit(Xtr_lx.loc[fm], counts_all[fm], eval_set=[(Xtr_lx.loc[vm], counts_all[vm])], verbose=False)
        bi = getattr(m, "best_iteration", None)
        bi = int(bi + 1) if bi is not None else XGB_PARAMS["n_estimators"]; iters.append(bi)
        pc = np.clip(np.asarray(m.predict(Xtr_lx.loc[vm]), float), 0, None)
        oof_true.append(counts_all[vm]); oof_pred.append(pc)
        _, q = tune_thresholds(to_class(counts_all[vm]), pc)
        log(f"[XGB] fold {k+1}/{N_FOLDS} best_iter={bi} QWK={q:.4f}")
    oof_true, oof_pred = np.concatenate(oof_true), np.concatenate(oof_pred)
    n_best = int(np.median(iters) * 1.10); log("[XGB] refit n_estimators", n_best)
    tp = np.zeros(len(Xte_lx))
    for sd in SEEDS:
        p = dict(XGB_PARAMS); p["n_estimators"] = n_best
        m = xgb.XGBRegressor(random_state=sd, **p)
        m.fit(Xtr_lx, counts_all, verbose=False)
        tp += np.asarray(m.predict(Xte_lx), float)
    save_preds("xgb", oof_true, oof_pred, np.clip(tp / len(SEEDS), 0, None))
    return load_saved("xgb")

best_xgb = get_model("xgb", train_xgb)

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [03:49:12] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[ 100.6s] [XGB] fold 1/3 best_iter=2359 QWK=0.7277
[ 187.3s] [XGB] fold 2/3 best_iter=2633 QWK=0.7428
[ 265.3s] [XGB] fold 3/3 best_iter=2228 QWK=0.7495
[ 265.3s] [XGB] refit n_estimators 2594
[ 520.3s] saved xgb_oof.parquet / xgb_test.parquet


### 4.3 Model 3 — CatBoost (RMSE on log count, keeps high-card IDs)

In [ ]:
CB_PARAMS = dict(loss_function="RMSE", eval_metric="RMSE", iterations=4000, learning_rate=0.03,
                 depth=8, l2_leaf_reg=20.0, random_strength=1.0, task_type=CAT_TASK_TYPE,
                 allow_writing_files=False, verbose=False)
if CAT_TASK_TYPE == "CPU":
    CB_PARAMS.update(bootstrap_type="Bernoulli", subsample=0.8, thread_count=-1)
else:  # GPU: Bayesian bootstrap uses bagging_temperature instead of subsample
    CB_PARAMS.update(bootstrap_type="Bayesian", bagging_temperature=1.0, devices="0")

def train_cb():
    oof_true, oof_pred, iters = [], [], []
    for k in range(N_FOLDS):
        fm, vm = fold_masks(k)
        trp = Pool(Xtr_cb.loc[fm], y_log_all[fm], cat_features=CATS_CB)
        vap = Pool(Xtr_cb.loc[vm], y_log_all[vm], cat_features=CATS_CB)
        m = CatBoostRegressor(random_seed=42, **CB_PARAMS)
        m.fit(trp, eval_set=vap, use_best_model=True, early_stopping_rounds=80, verbose=False)
        bi = int(m.get_best_iteration() or CB_PARAMS["iterations"]); iters.append(bi)
        pc = np.clip(np.expm1(m.predict(vap)), 0, None)
        oof_true.append(counts_all[vm]); oof_pred.append(pc)
        _, q = tune_thresholds(to_class(counts_all[vm]), pc)
        log(f"[CatBoost] fold {k+1}/{N_FOLDS} best_iter={bi} QWK={q:.4f}")
        del trp, vap, m; gc.collect()
    oof_true, oof_pred = np.concatenate(oof_true), np.concatenate(oof_pred)
    n_best = int(np.median(iters) * 1.10); log("[CatBoost] refit iterations", n_best)
    full = Pool(Xtr_cb, y_log_all, cat_features=CATS_CB)
    tepool = Pool(Xte_cb, cat_features=CATS_CB)
    tp = np.zeros(len(Xte_cb))
    for sd in SEEDS:
        p = dict(CB_PARAMS); p["iterations"] = n_best
        m = CatBoostRegressor(random_seed=sd, **p)
        m.fit(full, verbose=False)
        tp += np.expm1(m.predict(tepool))
    save_preds("cb", oof_true, oof_pred, np.clip(tp / len(SEEDS), 0, None))
    return load_saved("cb")

best_cb = get_model("cb", train_cb)

[ 558.8s] [CatBoost] fold 1/3 best_iter=919 QWK=0.6761
[ 603.9s] [CatBoost] fold 2/3 best_iter=1075 QWK=0.7186
[ 666.3s] [CatBoost] fold 3/3 best_iter=1498 QWK=0.7322
[ 666.4s] [CatBoost] refit iterations 1182
[ 805.7s] saved cb_oof.parquet / cb_test.parquet


## 5. Single-model scoreboard

QWK of each model on the pooled OOF and on the most-recent fold (our future-test proxy). Thresholds are tuned on the relevant window.

In [ ]:
MODELS = {"lgbm": best_lgb, "xgboost": best_xgb, "catboost": best_cb}
names = list(MODELS)
base_true = MODELS[names[0]]["oof_true"]
for n in names:
    assert np.array_equal(MODELS[n]["oof_true"], base_true), f"OOF order mismatch: {n}"
true_all, true_last = base_true, base_true[-N_LAST:]
OOF = np.vstack([MODELS[n]["oof_pred"] for n in names])
TST = np.vstack([MODELS[n]["test"] for n in names])

rows = []
for n in names:
    _, qo = tune_thresholds(to_class(true_all), MODELS[n]["oof_pred"])
    _, ql = tune_thresholds(to_class(true_last), MODELS[n]["oof_pred"][-N_LAST:])
    rows.append({"model": n, "OOF_QWK": round(qo, 4), "last_fold_QWK": round(ql, 4)})
scoreboard = pd.DataFrame(rows).sort_values("last_fold_QWK", ascending=False)
print(scoreboard.to_string(index=False))

   model  OOF_QWK  last_fold_QWK
 xgboost   0.7410         0.7495
    lgbm   0.7336         0.7463
catboost   0.7174         0.7322


## 6. Blend the three models and choose thresholds

We compare an **equal-weight** average against **non-negative weights optimised** to
balance pooled-OOF and last-fold QWK (50/50). The final 3 thresholds are tuned on the
most-recent fold — the closest proxy to the future test window.

In [ ]:
def eval_weights(w):
    w = np.clip(np.asarray(w, float), 0, None); w = w / (w.sum() or 1.0)
    ob = w @ OOF
    _, qo = tune_thresholds(to_class(true_all), ob)
    thr_last, ql = tune_thresholds(to_class(true_last), ob[-N_LAST:])
    return qo, ql, thr_last, w

w_eq = np.ones(len(names)) / len(names)
qo_e, ql_e, thr_e, w_eq = eval_weights(w_eq)
print("Equal weights   :", {n: round(float(x), 3) for n, x in zip(names, w_eq)})
print(f"  OOF_QWK={qo_e:.4f} | last_fold_QWK={ql_e:.4f} | thr={np.round(thr_e, 4)}")

def objective(w):
    qo, ql, _, _ = eval_weights(w); return -(0.5 * qo + 0.5 * ql)
r = minimize(objective, w_eq, method="Nelder-Mead", options={"xatol": 1e-3, "fatol": 1e-4, "maxiter": 4000})
qo_o, ql_o, thr_o, w_o = eval_weights(r.x)
print("Optimized weights:", {n: round(float(x), 3) for n, x in zip(names, w_o)})
print(f"  OOF_QWK={qo_o:.4f} | last_fold_QWK={ql_o:.4f} | thr={np.round(thr_o, 4)}")

Equal weights   : {'lgbm': 0.333, 'xgboost': 0.333, 'catboost': 0.333}
  OOF_QWK=0.7420 | last_fold_QWK=0.7506 | thr=[0.6324 1.2952 2.8368]
Optimized weights: {'lgbm': 0.319, 'xgboost': 0.341, 'catboost': 0.34}
  OOF_QWK=0.7430 | last_fold_QWK=0.7517 | thr=[0.603  1.3629 2.7672]


## 7. Write the final submission

Two files are written to both `RESULTS/` and `final_submission/`:
- `submission_final.csv` — the optimised blend (our chosen submission),
- `submission_ensemble_equal.csv` — the conservative equal-weight average.

We assert the format matches the phase-2 sample submission exactly (`id,target`, all ids
present and unique, `target ∈ {0,1,2,3}`).

In [ ]:
def write_sub(fname, w, thr):
    blend_test = np.clip(w @ TST, 0, None)
    pred = round_with(thr, blend_test).astype(int)
    sub = pd.DataFrame({"id": test_ids, "target": pred})
    assert len(sub) == len(test_ids) and sub["id"].is_unique
    assert set(np.unique(sub["target"])) <= {0, 1, 2, 3}
    sub.to_csv(os.path.join(SUB_DIR, fname), index=False)
    sub.to_csv(os.path.join(RESULTS, fname), index=False)
    d = sub["target"].value_counts(normalize=True).sort_index().round(4).to_dict()
    print(f"wrote {fname}  dist={d}")
    return sub

sub_equal = write_sub("submission_ensemble_equal.csv", w_eq, thr_e)
sub_final = write_sub("submission_final.csv", w_o, thr_o)

# Validate against the official phase-2 sample submission.
sample = pd.read_csv(os.path.join(TEST_DIR, "test_phase2_sample_submission.csv"))
assert set(sub_final["id"]) == set(sample["id"]), "id set does not match the sample submission"
assert list(sub_final.columns) == ["id", "target"]

json.dump({"oof_qwk_equal": float(qo_e), "last_qwk_equal": float(ql_e),
           "oof_qwk_opt": float(qo_o), "last_qwk_opt": float(ql_o),
           "weights_opt": {n: float(x) for n, x in zip(names, w_o)},
           "thr_opt": [float(x) for x in thr_o]},
          open(os.path.join(RESULTS, "blend_meta.json"), "w"), indent=2)

print("\nFINAL submission ->", os.path.join(SUB_DIR, "submission_final.csv"))
sub_final.head()

wrote submission_ensemble_equal.csv  dist={0: 0.5752, 1: 0.1933, 2: 0.1876, 3: 0.0438}
wrote submission_final.csv  dist={0: 0.5557, 1: 0.2262, 2: 0.1705, 3: 0.0476}

FINAL submission -> /content/drive/MyDrive/Prozzoro/final_submission/submission_final.csv


,id,target
0,PZ9000000,0
1,PZ9000001,1
2,PZ9000002,2
3,PZ9000003,0
4,PZ9000004,0


## 8. Decision summary

- **Submit `submission_final.csv`** (optimised blend) when its last-fold / OOF QWK clearly
  beats the equal average *and* the single models, and the predicted class distribution is
  not skewed. Otherwise fall back to `submission_ensemble_equal.csv`.
- Everything is cached in `RESULTS/`: delete a `{tag}_oof/_test.parquet` pair to retrain
  just that model, or delete `features_*.parquet` to rebuild features from scratch.

In [ ]:
final_overview = pd.DataFrame([
    {"submission": "equal",     "OOF_QWK": round(qo_e, 4), "last_fold_QWK": round(ql_e, 4)},
    {"submission": "optimized", "OOF_QWK": round(qo_o, 4), "last_fold_QWK": round(ql_o, 4)},
])
print(final_overview.to_string(index=False))
print("\nChosen submission: submission_final.csv  (optimized blend)")
print("Predicted test distribution:",
      sub_final["target"].value_counts(normalize=True).sort_index().round(4).to_dict())

submission  OOF_QWK  last_fold_QWK
     equal    0.742         0.7506
 optimized    0.743         0.7517

Chosen submission: submission_final.csv  (optimized blend)
Predicted test distribution: {0: 0.5557, 1: 0.2262, 2: 0.1705, 3: 0.0476}


In [ ]:
sub_final.shape

(2100, 2)